In [1]:
import sys
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import pcse
import numpy as np

In [ ]:
# Define Paths
project = Path.cwd().parents[1]

soil_file = project / "marginal_simulations_eu" / "soil" / "soil_EC1-coarse.soil"
agro_file = project / "marginal_simulations_eu" / "agromanager" / "misc_agro.yaml"
#weather_folder = project / "marginal_simulations_eu" / "weather" / "agera5_csv"
crop_file = project / "sinensis" / "miscanthus_calibration" / "input_params" / "miscanthus" / "miscanthus.yaml"

In [3]:
# Load soil & crop parameters
from pcse.input import YAMLCropDataProvider

cropdata = YAMLCropDataProvider(fpath=crop_file.parent, force_reload=True)
cropdata.set_active_crop("miscanthus", "miscanthus_sinensis")
print(cropdata)

from pcse.input import CABOFileReader

soildata = CABOFileReader(soil_file)
print(soildata)

# Derive initial amount of water in soil profile 
SMFCF = soildata["SMFCF"]
SMW   = soildata["SMW"]
RDI   = cropdata["RDI"]        # initial rooting depth (cm)
wav_init = 0.5 * (SMFCF - SMW) * RDI * 25   # mm
print(f"Initial WAV = {wav_init:.1f} mm  (50% between FC and WP)")

from pcse.input import WOFOST72SiteDataProvider
sitedata = WOFOST72SiteDataProvider(WAV=wav_init)

print(sitedata)

# Package all parameters
from pcse.base import ParameterProvider

parameters = ParameterProvider(cropdata=cropdata, soildata=soildata, sitedata=sitedata)

Crop parameters loaded from: /Users/paulianoprescu/Projects/wofost_miscanthus/sinensis/miscanthus_calibration/input_params/miscanthus
YAMLCropDataProvider - current active crop 'miscanthus' with variety 'miscanthus_sinensis'
Available crop parameters:
 {'CO2EFFTB': [40.0, 0.0, 360.0, 1.0, 720.0, 1.0, 1000.0, 1.0, 2000.0, 1.0], 'CO2TRATB': [40.0, 0.0, 360.0, 1.0, 720.0, 0.74, 1000.0, 0.74, 2000.0, 0.74], 'CO2AMAXTB': [40.0, 0.0, 360.0, 1.0, 720.0, 1.0, 1000.0, 1.0, 2000.0, 1.0], 'TBASEM': 8.0, 'TEFFMX': 30.0, 'TSUMEM': 0, 'IDSL': 0, 'DLO': 1.0, 'DLC': 0.0, 'TSUM1': 1796.93, 'TSUM2': 203.068, 'DTSMTB': [0.0, 0.0, 8.0, 0.0, 35.0, 29.0, 45.0, 29.0], 'DVSI': 0, 'DVSEND': 2.0, 'VERNBASE': 14.0, 'VERNSAT': 70.0, 'VERNRTB': [-8.0, 0.0, -4.0, 0.0, 3.0, 1.0, 10.0, 1.0, 17.0, 0.0, 20.0, 0.0], 'TDWI': 600.0, 'RGRLAI': 0.05, 'SLATB': [0.0, 0.0009, 0.21, 0.0019, 0.29, 0.0012, 0.64, 0.0008, 1.0, 0.001, 2.0, 0.001], 'SPA': 0.0, 'SSATB': [0.0, 0.0, 2.0, 9.98e-07], 'SPAN': 75.0, 'TBASE': 10.0, 'KDIFTB':

In [4]:
# Define CropCalendar and simulation Campaign
from pcse.input import YAMLAgroManagementReader

agromanagement = YAMLAgroManagementReader(agro_file)
print(agromanagement)

!!python/object/new:pcse.input.yaml_agro_loader.YAMLAgroManagementReader
listitems:
- 2018-10-01:
    CropCalendar:
      crop_end_date: 2019-12-30
      crop_end_type: harvest
      crop_name: miscanthus
      crop_start_date: 2019-01-01
      crop_start_type: emergence
      max_duration: null
      variety_name: miscanthus_sinensis
    StateEvents: null
    TimedEvents: null
- 2020-01-01:
    CropCalendar:
      crop_end_date: 2020-12-30
      crop_end_type: harvest
      crop_name: miscanthus
      crop_start_date: 2020-01-01
      crop_start_type: emergence
      max_duration: null
      variety_name: miscanthus_sinensis
    StateEvents: null
    TimedEvents: null
- 2021-01-01:
    CropCalendar:
      crop_end_date: 2021-12-30
      crop_end_type: harvest
      crop_name: miscanthus
      crop_start_date: 2021-01-01
      crop_start_type: emergence
      max_duration: null
      variety_name: miscanthus_sinensis
    StateEvents: null
    TimedEvents: null
- 2022-01-01:
    CropCal

In [5]:
# Cell 4 — load cached AgERA5 providers (built in weather_extraction_nuts3.ipynb)
import sys, pickle, zlib
sys.path.insert(0, str(project / "marginal_simulations_eu" / "weather"))  # find weather_provider + config
from weather_provider import WOFOSTWebWeatherDataProvider                 # needed to unpickle the WDPs
from sqlitedict import SqliteDict

def _decode(obj):
    return pickle.loads(zlib.decompress(bytes(obj)))

cache_file = project / "marginal_simulations_eu" / "weather" / "agera5_cache.sqlite"
weather_db = SqliteDict(str(cache_file), decode=_decode)
print(f"{len(weather_db)} cached weather providers")

465 cached weather providers


In [6]:
# Run Water-Limited Production Simulations for each NUTS3 region
from pcse.models import Wofost72_WLP_CWB

records = []
for nuts3 in weather_db.keys():
    try:
        weatherdata = weather_db[nuts3]
        wofost = Wofost72_WLP_CWB(parameters, weatherdata, agromanagement)
        wofost.run_till_terminate()
        df = pd.DataFrame(wofost.get_output()).set_index("day")
        df["year"] = df.index.map(lambda d: d.year)
        for year, tagp in (df.groupby("year")["TAGP"].max() / 1000).items():
            records.append({"nuts3": nuts3, "year": year, "yield_tha": tagp})
    except Exception as e:
        print(f"{nuts3:6s} FAILED -- {e}")

weather_db.close()
results = pd.DataFrame(records)
print(f"\nsimulated {results['nuts3'].nunique()} regions")


simulated 465 regions


In [7]:
# Make average yield (2019-2023) map
import geopandas as gpd

out_dir = project / "marginal_simulations_eu" / "spatial_results" / "sinensis"

avg = (results[results["year"].between(2019, 2023)]
       .groupby("nuts3")["yield_tha"].mean()
       .reset_index(name="yield_avg"))

# bring in the marginality factors per region
sites = pd.read_csv(project / "marginal_simulations_eu" / "sites_geopandas" / "nuts3_centroid.csv")
avg = avg.merge(sites[["nuts3", "constraints"]], on="nuts3", how="left")

results.to_csv(out_dir / "yield_per_year.csv", index=False)
avg.to_csv(out_dir / "yield_avg_2019_2023.csv", index=False)

shp = project / "marginal_simulations_eu" / "sites_geopandas" / "S2BiomeShape" / "S2B_1339_GDD_PPET.shp" 
poly = gpd.read_file(shp)
poly = poly[["NUTS_CODE", "geometry"]]                                                            
mask = poly.merge(avg, left_on="NUTS_CODE", right_on="nuts3", how="inner").drop(columns="nuts3")  
mask.to_file(out_dir / "yield_avg_2019_2023.gpkg", driver="GPKG")
print(f"saved {len(mask)} regions with yield + constraints")

saved 465 regions with yield + constraints


In [8]:
# Extract values for report
print(f"regions simulated : {len(avg)}")
print(f"mean yield        : {avg['yield_avg'].mean():.2f} t/ha")
print(f"regions < 5 t/ha  : {(avg['yield_avg'] < 5).sum()}")
print(f"highest           : {avg['yield_avg'].max():.2f} t/ha")
print(f"lowest            : {avg['yield_avg'].min():.2f} t/ha")
print("\ntop 5:")
print(avg.nlargest(5, "yield_avg")[["nuts3", "yield_avg", "constraints"]].to_string(index=False))
print("\nbottom 5:")
print(avg.nsmallest(5, "yield_avg")[["nuts3", "yield_avg", "constraints"]].to_string(index=False))

regions simulated : 465
mean yield        : 9.18 t/ha
regions < 5 t/ha  : 92
highest           : 23.51 t/ha
lowest            : 0.30 t/ha

top 5:
nuts3  yield_avg                                         constraints
ITH20  23.513454 slope; soil_depth; soil_moisture; frost_damage; gdd
ITC46  22.504577                                  soil_moisture; gdd
ITC11  21.767799                                                 gdd
ITH41  20.115825                         soil_moisture; frost_damage
ITC42  20.101195                           slope; soil_moisture; gdd

bottom 5:
nuts3  yield_avg                                         constraints
ITC44   0.298409 slope; soil_depth; soil_moisture; frost_damage; gdd
AT334   0.535737             slope; soil_moisture; frost_damage; gdd
AT331   0.784825 slope; soil_depth; soil_moisture; frost_damage; gdd
ES300   0.896295                                             dryness
EL306   0.973771                                          soil_depth
